# 04-2 · Per-diner dish compensation (C4)

| | |
|---|---|
| **What it does** | assigns each member their own best SAFE dish at the chosen restaurant |
| **Proves / tests H4** | per-person dish assignment raises the floor above the restaurant-level MinSat |
| **Source** | `nbio.py` |
| **Caution** | the before/after comparison is NOT apples-to-apples — flagged below, not hidden |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator')

## Rebuild the pick from 04-1


In [3]:
# Step 1: rebuild F, S, members, and the fair(0.5) pick, safe dishes attached
personas = json.load(open(nbio.SMALL_PERSONAS))
group_names = ["Ravi", "Marco", "Omar", "Alice"]
members = []
for name in group_names:
    p = next(x for x in personas if x["display_name"] == name)
    members.append({"member_id": name.lower(), "display_name": name,
                     "diet": p["diet_type"], "allergies": p["allergies"], "taste": p["taste_vector"]})

import csv
flavor_map = {}
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    for row in csv.DictReader(f):
        n = row.get("ingredient", "").strip().lower()
        if n: flavor_map[n] = {d: float(row.get(d, 0) or 0) for d in nbio.TASTE_DIMS}

def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip(); return
        for v in node.values(): yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node: yield from iter_dish_names(v)

def infer_taste(d):
    m = [flavor_map[i] for i in flavor_map if i in d.lower()]
    return [sum(x[k] for x in m)/len(m) for k in nbio.TASTE_DIMS] if m else [0.0]*6

catalog = []
for f in sorted(nbio.SMALL_MENUS.glob("*.json")):
    names = [n for n in iter_dish_names(json.loads(f.read_text())) if n]
    catalog.append({"option_id": f.stem, "name": f.stem,
                     "menu": [{"name": n, "taste": infer_taste(n)} for n in names]})
for o in catalog:
    nz = [d["taste"] for d in o["menu"] if sum(abs(x) for x in d["taste"]) > 0]
    o["taste"] = [sum(v[i] for v in nz)/len(nz) for i in range(6)] if nz else [0.0]*6

F = nbio.build_feasible_set(catalog, members)
S = {o["option_id"]: {m["member_id"]: nbio.taste_similarity(m["taste"], o["taste"]) for m in members} for o in F}
candidate_vectors = {o["option_id"]: o["taste"] for o in F}
member_vectors = [m["taste"] for m in members]
labels = {o["option_id"]: o["name"] for o in F}

pick_id, _ = nbio.rank_candidates(candidate_vectors, member_vectors, method="fair", lam=0.5)[0]
pick = next(o for o in F if o["option_id"] == pick_id)
print("pick:", labels[pick_id])


pick: Casa.cafe


## Assign each member their own best safe dish


In [4]:
# Step 2: for each member, the highest-similarity dish from their own safe set
dish_for = {}
for m in members:
    safe = pick["safe_dishes"][m["member_id"]]
    if not safe:
        continue
    best = max(safe, key=lambda d: nbio.taste_similarity(m["taste"], d["taste"]))
    dish_for[m["member_id"]] = {"name": best["name"],
                                 "similarity": nbio.taste_similarity(m["taste"], best["taste"]),
                                 "n_safe_dishes": len(safe)}

nbio.table([{"member": m["display_name"], "s_i on restaurant": S[pick_id][m["member_id"]],
             "their dish": dish_for[m["member_id"]]["name"][:28],
             "dish sim": dish_for[m["member_id"]]["similarity"],
             "safe pool": dish_for[m["member_id"]]["n_safe_dishes"]}
            for m in members if m["member_id"] in dish_for],
           cols=["member", "s_i on restaurant", "their dish", "dish sim", "safe pool"], width=28)


member  s_i on restaurant  their dish                   dish sim  safe pool
---------------------------------------------------------------------------
Ravi    0.624              Veg Spring Roll              0.955     70       
Marco   0.746              Veg Spring Roll              0.899     44       
Omar    0.825              Mocha Milkshake with Coffee  0.933     129      
Alice   0.849              Veg Spring Roll              0.862     129      


## The compensation claim, measured


In [5]:
# Step 3: floor before vs after compensation
restaurant_min = min(S[pick_id][m["member_id"]] for m in members)
dish_min = min(d["similarity"] for d in dish_for.values())
print(f"floor BEFORE (restaurant-level MinSat)  : {restaurant_min:.3f}")
print(f"floor AFTER  (compensated MinSat)        : {dish_min:.3f}")
print(f"change                                    : {dish_min - restaurant_min:+.3f}")


floor BEFORE (restaurant-level MinSat)  : 0.624
floor AFTER  (compensated MinSat)        : 0.862
change                                    : +0.238


> **Caution, stated rather than hidden.** These two numbers are not on the same
> scale. `restaurant_min` is cosine against a restaurant's WHOLE-MENU average
> taste vector; `dish_min` is cosine against ONE dish hand-picked to match a
> single member. A dish chosen to fit one person will almost always score
> higher than any restaurant average — some of the apparent lift is a property
> of the comparison, not of fairness recovered. Report it with this caveat, or
> construct a like-for-like baseline.


### Reading it

The restaurant vote has one winner; the dish assignment has n. Whatever the
scale-mismatch caveat above, the mechanism itself is real: a member who
"loses" the restaurant vote is not stuck with the restaurant's average dish —
they get their own best safe option. **H4** in the paper's plan asks whether
this measurably helps the worst-off member; this notebook is the harness for
that question, honestly caveated.

Next: **05-mediate** — the closed-loop mediator and veto token.


## Revision: the experienced (dish-level) outcome

The venue-level satisfaction above is a proxy: a diner eats one dish, not the
whole menu. This section scores what each diner would actually experience, their
best safe dish at the chosen restaurant (the order-aware utility `D`), for picks
made by different pipelines: Average, Fair (lambda 0.5 and 5) and Least-Misery
selecting on the whole-menu proxy, and Average, Fair (lambda 5) and Least-Misery
selecting directly on the order-aware utility. Food.com diners on Uber Eats
Houston and Seattle, 200 seeded groups each (`nbio.rev_data()`).

Reports the mean experienced MinSat (with a 95% bootstrap CI) and mean, and the
paired gain of order-aware Average over the paper's pipeline (Fair, lambda 0.5,
on the proxy). Writes `figdata_rev/e2b_experienced.csv`; the last cell writes the
LaTeX table rows `figdata_rev/tex/tab_experienced.tex` from it.

In [6]:
import csv
import numpy as np
from nbio import REV_CITIES, rev_data, rev_pick, rev_boot_ci

D = rev_data()
rows=[]
for city in REV_CITIES:
    G=D[city]
    pipes=[("Average on whole-menu proxy","S","average",0),("Fair (lambda=0.5) on whole-menu proxy","S","fair",0.5),
           ("Fair (lambda=5) on whole-menu proxy","S","fair",5),("Least-Misery on whole-menu proxy","S","least_misery",0),
           ("Average on order-aware utility","D","average",0),("Fair (lambda=5) on order-aware utility","D","fair",5),
           ("Least-Misery on order-aware utility","D","least_misery",0)]
    for name,key,rule,lam in pipes:
        ps=[rev_pick(g[key],rule,lam) for g in G]
        mins=[g["D"][p].min() for g,p in zip(G,ps)]; means=[g["D"][p].mean() for g,p in zip(G,ps)]
        lo,hi=rev_boot_ci(mins)
        rows.append(dict(city=city,pipeline=name,exp_minsat=round(np.mean(mins),4),lo=round(lo,4),hi=round(hi,4),exp_mean=round(np.mean(means),4)))
        print(rows[-1])
    # paired gain order-aware Average vs paper pipeline Fair(0.5,S)
    a=[g["D"][rev_pick(g["D"],"average")].min()-g["D"][rev_pick(g["S"],"fair",0.5)].min() for g in G]
    print(city,"paired gain",np.mean(a),rev_boot_ci(a), "groups worse:",sum(x< -1e-12 for x in a))
with open(nbio.REV_OUT + "/e2b_experienced.csv","w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

{'city': 'Houston, TX', 'pipeline': 'Average on whole-menu proxy', 'exp_minsat': np.float64(0.9492), 'lo': 0.9425, 'hi': 0.9553, 'exp_mean': np.float64(0.9729)}
{'city': 'Houston, TX', 'pipeline': 'Fair (lambda=0.5) on whole-menu proxy', 'exp_minsat': np.float64(0.9477), 'lo': 0.9409, 'hi': 0.9538, 'exp_mean': np.float64(0.9724)}
{'city': 'Houston, TX', 'pipeline': 'Fair (lambda=5) on whole-menu proxy', 'exp_minsat': np.float64(0.9477), 'lo': 0.9418, 'hi': 0.9531, 'exp_mean': np.float64(0.9733)}
{'city': 'Houston, TX', 'pipeline': 'Least-Misery on whole-menu proxy', 'exp_minsat': np.float64(0.9444), 'lo': 0.9378, 'hi': 0.9504, 'exp_mean': np.float64(0.9721)}
{'city': 'Houston, TX', 'pipeline': 'Average on order-aware utility', 'exp_minsat': np.float64(0.9855), 'lo': 0.9846, 'hi': 0.9864, 'exp_mean': np.float64(0.9919)}
{'city': 'Houston, TX', 'pipeline': 'Fair (lambda=5) on order-aware utility', 'exp_minsat': np.float64(0.9855), 'lo': 0.9847, 'hi': 0.9864, 'exp_mean': np.float64(0.9919

Houston, TX paired gain 0.03783324750032267 (0.03195555962588338, 0.04430975510438549) groups worse: 2
{'city': 'Seattle, WA', 'pipeline': 'Average on whole-menu proxy', 'exp_minsat': np.float64(0.9451), 'lo': 0.9377, 'hi': 0.9516, 'exp_mean': np.float64(0.9732)}
{'city': 'Seattle, WA', 'pipeline': 'Fair (lambda=0.5) on whole-menu proxy', 'exp_minsat': np.float64(0.9439), 'lo': 0.9362, 'hi': 0.9507, 'exp_mean': np.float64(0.9727)}
{'city': 'Seattle, WA', 'pipeline': 'Fair (lambda=5) on whole-menu proxy', 'exp_minsat': np.float64(0.9438), 'lo': 0.9363, 'hi': 0.9504, 'exp_mean': np.float64(0.9719)}
{'city': 'Seattle, WA', 'pipeline': 'Least-Misery on whole-menu proxy', 'exp_minsat': np.float64(0.9442), 'lo': 0.9384, 'hi': 0.9496, 'exp_mean': np.float64(0.9707)}
{'city': 'Seattle, WA', 'pipeline': 'Average on order-aware utility', 'exp_minsat': np.float64(0.9847), 'lo': 0.9834, 'hi': 0.9859, 'exp_mean': np.float64(0.9915)}
{'city': 'Seattle, WA', 'pipeline': 'Fair (lambda=5) on order-awar

{'city': 'Seattle, WA', 'pipeline': 'Least-Misery on order-aware utility', 'exp_minsat': np.float64(0.9864), 'lo': 0.9854, 'hi': 0.9874, 'exp_mean': np.float64(0.9908)}
Seattle, WA paired gain 0.04086258448193191 (0.03438365596663577, 0.048166209374790346) groups worse: 0


In [7]:
# LaTeX table rows (tab_experienced.tex) from the CSV above
import pandas as pd
R = nbio.REV_OUT
T = os.path.join(nbio.REV_OUT, "tex")
os.makedirs(T, exist_ok=True)
H, S = "Houston, TX", "Seattle, WA"


def f(x, d=4):
    return f"{x:.{d}f}"


e = pd.read_csv(f"{R}/e2b_experienced.csv")
lines = []
for i, p in enumerate(e[e.city == H].pipeline):
    if i == 4:
        lines.append(r"\midrule")
    cells = [p.replace("lambda=", r"$\lambda{=}").replace(")", "$)") if "lambda" in p else p]
    for c in (H, S):
        r = e[(e.city == c) & (e.pipeline == p)].iloc[0]
        cells += [f"{r.exp_minsat:.4f} [{r.lo:.4f}, {r.hi:.4f}]", f(r.exp_mean)]
    lines.append(" & ".join(cells) + r" \\")
open(f"{T}/tab_experienced.tex", "w").write("\n".join(lines) + "\n")
print(open(f"{T}/tab_experienced.tex").read())

Average on whole-menu proxy & 0.9492 [0.9425, 0.9553] & 0.9729 & 0.9451 [0.9377, 0.9516] & 0.9732 \\
Fair ($\lambda{=}0.5$) on whole-menu proxy & 0.9477 [0.9409, 0.9538] & 0.9724 & 0.9439 [0.9362, 0.9507] & 0.9727 \\
Fair ($\lambda{=}5$) on whole-menu proxy & 0.9477 [0.9418, 0.9531] & 0.9733 & 0.9438 [0.9363, 0.9504] & 0.9719 \\
Least-Misery on whole-menu proxy & 0.9444 [0.9378, 0.9504] & 0.9721 & 0.9442 [0.9384, 0.9496] & 0.9707 \\
\midrule
Average on order-aware utility & 0.9855 [0.9846, 0.9864] & 0.9919 & 0.9847 [0.9834, 0.9859] & 0.9915 \\
Fair ($\lambda{=}5$) on order-aware utility & 0.9855 [0.9847, 0.9864] & 0.9919 & 0.9848 [0.9835, 0.9859] & 0.9915 \\
Least-Misery on order-aware utility & 0.9871 [0.9863, 0.9878] & 0.9910 & 0.9864 [0.9854, 0.9874] & 0.9908 \\

